In [1]:
from IPython import get_ipython
from IPython.display import display
# %% [markdown]
# # Pandas for Data Science & Visualization
# 
# %% [markdown]
# ## Setup
# Colab already has pandas, numpy, matplotlib and seaborn installed — just import them.
# %%
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
# %% [markdown]
# # Part 1 — Pandas Basics
# 
# Pandas has two core data structures:
# - **`Series`** → a single column of data (1D, labeled array)
# - **`DataFrame`** → a table of data (2D, rows + columns), basically a collection of Series sharing an index
# %% [markdown]
# ### 1.1 Creating a Series
# %%
s = pd.Series([10, 20, 30, 40], name="marks")
print(s)
print()
print("Values:", s.values)
print("Index:", s.index.tolist())
# %%
# Series with a custom index (like a dictionary with order + vectorized ops)
s2 = pd.Series([85, 90, 78], index=["Amit", "Priya", "Rahul"], name="score")
s2
# %% [markdown]
# ### 1.2 Creating a DataFrame
# %%
# 1) From a dictionary of lists (most common way)
data = {
    "name": ["Amit", "Priya", "Rahul", "Sneha", "Vikram"],
    "age": [23, 21, 25, 22, 24],
    "city": ["Delhi", "Mumbai", "Pune", "Delhi", "Chennai"],
    "marks": [85, 92, 78, 88, 95]
}
df = pd.DataFrame(data)
df
# %%
# 2) From a list of dictionaries (row-wise)
rows = [
    {"name": "Amit", "age": 23},
    {"name": "Priya", "age": 21},
]
pd.DataFrame(rows)
# %%
# 3) From a numpy array
arr = np.random.randint(1, 100, size=(4, 3))
pd.DataFrame(arr, columns=["A", "B", "C"])
# %% [markdown]
# ### 1.3 Exploring a DataFrame
# %%
print(df.shape)      # (rows, columns)
print(df.columns.tolist())
print(df.index.tolist())
print(df.dtypes)
# %%
df.head(3)      # first 3 rows
# %%
df.tail(2)      # last 2 rows
# %%
df.sample(2)    # 2 random rows
# %%
df.info()       # column names, non-null counts, dtypes, memory usage
# %%
df.describe()   # summary statistics for numeric columns
# %% [markdown]
# > 🧠 **Tip:** `.info()` tells you about *structure* (types, missing values). `.describe()` tells you about *distribution* (mean, std, min/max, quartiles). Always run both when you get a new dataset.
# %% [markdown]
# # Part 2 — Reading & Writing Data
# 
# In real projects, data usually comes from files (CSV, Excel, JSON, SQL) rather than being typed by hand.
# %%
# Save our DataFrame to a CSV file, then read it back
df.to_csv("students.csv", index=False)

df_csv = pd.read_csv("students.csv")
df_csv
# %%
# JSON
df.to_json("students.json", orient="records")
df_json = pd.read_json("students.json")
df_json
# %% [markdown]
# Other common readers/writers you'll use outside this notebook:
# 
# ```python
# pd.read_excel("file.xlsx", sheet_name="Sheet1")
# df.to_excel("file.xlsx", index=False)
# 
# pd.read_sql("SELECT * FROM table", connection)
# 
# pd.read_csv("data.csv", sep=";", encoding="utf-8", na_values=["NA", "?"])
# ```
# %%
# Converting to other Python structures
print(df.to_dict(orient="records")[:2])
print()
print(df.to_numpy()[:2])
# %% [markdown]
# # Part 3 — Selection & Indexing
# 
# This is the single most-used skill in pandas: picking the rows/columns you need.
# %% [markdown]
# ### 3.1 Selecting columns
# %%
df["name"]              # single column -> Series
# %%
df[["name", "marks"]]   # multiple columns -> DataFrame
# %% [markdown]
# ### 3.2 Selecting rows — `.loc` (label-based) vs `.iloc` (position-based)
# %%
df.loc[0]            # row with index label 0
# %%
df.loc[0:2]           # rows with labels 0,1,2 (inclusive!)
# %%
df.iloc[0:2]          # rows at positions 0,1 (exclusive of 2, like normal Python slicing)
# %%
df.loc[0:2, ["name", "city"]]   # rows 0-2, specific columns
# %%
df.iloc[1, 0]          # value at row-position 1, column-position 0
# %% [markdown]
# ### 3.3 Boolean (conditional) indexing
# %%
df[df["marks"] > 85]
# %%
# Multiple conditions: use & (and), | (or), ~ (not) — and wrap each condition in ( )
df[(df["marks"] > 80) & (df["city"] == "Delhi")]
# %%
# .query() -- often more readable for complex conditions
df.query("marks > 80 and city == 'Delhi'")
# %% [markdown]
# ### 3.4 Adding, modifying, dropping columns
# %%
df["grade"] = np.where(df["marks"] >= 90, "A", np.where(df["marks"] >= 80, "B", "C"))
df
# %%
df.drop(columns=["grade"])     # returns a NEW dataframe (original untouched unless inplace=True)
# %% [markdown]
# ### 3.5 Setting a custom index
# %%
df_indexed = df.set_index("name")
df_indexed
# %%
df_indexed.loc["Priya"]        # now you can select rows by name directly
# %%
df_indexed.reset_index()       # undo set_index, back to default 0..n index
# %% [markdown]
# # Part 4 — Data Cleaning
# 
# Real-world data is messy: missing values, duplicates, wrong types, inconsistent text. Let's build a messy dataset on purpose and clean it.
# %%
messy = pd.DataFrame({
    "name": ["Amit", "Priya", "Rahul", "Rahul", "Sneha", None],
    "age": [23, np.nan, 25, 25, 22, 30],
    "city": ["Delhi", "mumbai", "Pune", "Pune", "DELHI", "Chennai"],
    "salary": ["50000", "62,000", "58000", "58000", "N/A", "71000"]
})
messy
# %% [markdown]
# ### 4.1 Detecting missing values
# %%
print(messy.isna().sum())     # count of missing values per column
messy.isna()                  # boolean mask, True = missing
# %% [markdown]
# ### 4.2 Handling missing values
# %%
messy.dropna(subset=["name"])          # drop rows where 'name' is missing
# %%
messy_filled = messy.copy()
messy_filled["age"] = messy_filled["age"].fillna(messy_filled["age"].median())
messy_filled
# %% [markdown]
# ### 4.3 Removing duplicates
# %%
print(messy.duplicated().sum())        # how many fully duplicate rows?
messy.drop_duplicates()
# %% [markdown]
# ### 4.4 Fixing text inconsistencies
# %%
messy_clean = messy.copy()
messy_clean["city"] = messy_clean["city"].str.strip().str.title()
messy_clean
# %% [markdown]
# ### 4.5 Fixing data types
# %%
messy_clean["salary"] = (
    messy_clean["salary"]
    .replace("N/A", np.nan)
    .str.replace(",", "", regex=False)
    .astype(float)
)
messy_clean.dtypes
# %% [markdown]
# ### 4.6 Renaming columns & replacing values
# %%
messy_clean = messy_clean.rename(columns={"salary": "salary_inr"})
messy_clean["city"] = messy_clean["city"].replace({"Delhi": "New Delhi"})
messy_clean
# %% [markdown]
# > 🧠 **Cleaning checklist for any new dataset:**
# > 1. `.info()` → check dtypes & missing counts
# > 2. `.duplicated().sum()` → check duplicates
# > 3. `.describe()` → check for impossible values (negative age, etc.)
# > 4. Standardize text (`.str.strip()`, `.str.lower()`/`.str.title()`)
# > 5. Fix dtypes (`astype`) *after* cleaning text representations of numbers
# %% [markdown]
# # Part 5 — Data Manipulation
# 
# Transforming columns, applying custom functions, sorting, binning, and working with text/dates.
# %% [markdown]
# ### 5.1 `apply()`, `map()`
# %%
df["marks_percent"] = df["marks"].apply(lambda x: f"{x}%")
df
# %%
# map() works element-wise on a Series, great for simple lookups
city_region = {"Delhi": "North", "Mumbai": "West", "Pune": "West", "Chennai": "South"}
df["region"] = df["city"].map(city_region)
df
# %%
# apply() on a whole row (axis=1) to combine multiple columns
df["summary"] = df.apply(lambda row: f"{row['name']} ({row['age']}) - {row['marks']} marks", axis=1)
df[["name", "summary"]]
# %% [markdown]
# ### 5.2 Sorting
# %%
df.sort_values("marks", ascending=False)
# %%
df.sort_values(["city", "marks"], ascending=[True, False])   # sort by multiple columns
# %% [markdown]
# ### 5.3 Ranking
# %%
df["rank"] = df["marks"].rank(ascending=False).astype(int)
df.sort_values("rank")
# %% [markdown]
# # Part 6 — GroupBy & Aggregation
# 
# `groupby()` follows the **split → apply → combine** pattern: split data into groups, apply a function to each group, combine the results.
# %%
sales = pd.DataFrame({
    "region": ["North", "South", "North", "West", "South", "West", "North"],
    "product": ["A", "B", "A", "A", "A", "B", "B"],
    "units": [10, 5, 8, 12, 7, 9, 4],
    "revenue": [1000, 500, 800, 1200, 700, 900, 400]
})
sales
# %% [markdown]
# ### 6.1 Basic groupby + aggregation
# %%
sales.groupby("region")["revenue"].sum()
# %%
sales.groupby("region")["revenue"].mean().round(2)
# %% [markdown]
# ### 6.2 Multiple aggregations with `.agg()`
# %%
sales.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_units=("units", "mean"),
    orders=("units", "count")
).reset_index()
# %% [markdown]
# ### 6.3 Grouping by multiple columns
# %%
sales.groupby(["region", "product"])["revenue"].sum().reset_index()
# %% [markdown]
# ### 6.4 `transform()` — add group stats back onto every row
# %%
sales["region_avg_revenue"] = sales.groupby("region")["revenue"].transform("mean")
sales
# %% [markdown]
# ### 6.5 `filter()` — keep only groups matching a condition
# %%
sales.groupby("region").filter(lambda g: g["revenue"].sum() > 1500)
# %% [markdown]
# # Part 8 — Reshaping Data: `pivot_table`, `melt`, `crosstab`
# 
# Reshaping switches data between **wide** format (one row per entity, many columns) and **long/tidy** format (one row per observation).
# %% [markdown]
# ### 8.1 `pivot_table` — long → wide, with aggregation
# %%
sales.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum", fill_value=0)
# %% [markdown]
# ### 8.2 `melt` — wide → long
# %%
wide = pd.DataFrame({
    "student": ["Amit", "Priya"],
    "math": [80, 90],
    "science": [75, 95]
})
wide.melt(id_vars="student", var_name="subject", value_name="score")


In [3]:
# Pandas for Data Science & Visualization


In [4]:
## Setup
Colab already has pandas, numpy, matplotlib and seaborn installed — just import them.

In [5]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

In [6]:
# Part 1 — Pandas Basics

Pandas has two core data structures:
- **`Series`** → a single column of data (1D, labeled array)
- **`DataFrame`** → a table of data (2D, rows + columns), basically a collection of Series sharing an index

In [7]:
### 1.1 Creating a Series

In [8]:
s = pd.Series([10, 20, 30, 40], name="marks")
print(s)
print()
print("Values:", s.values)
print("Index:", s.index.tolist())

In [9]:
# Series with a custom index (like a dictionary with order + vectorized ops)
s2 = pd.Series([85, 90, 78], index=["Amit", "Priya", "Rahul"], name="score")
s2

In [10]:
### 1.2 Creating a DataFrame

In [11]:
# 1) From a dictionary of lists (most common way)
data = {
    "name": ["Amit", "Priya", "Rahul", "Sneha", "Vikram"],
    "age": [23, 21, 25, 22, 24],
    "city": ["Delhi", "Mumbai", "Pune", "Delhi", "Chennai"],
    "marks": [85, 92, 78, 88, 95]
}
df = pd.DataFrame(data)
df

In [12]:
# 2) From a list of dictionaries (row-wise)
rows = [
    {"name": "Amit", "age": 23},
    {"name": "Priya", "age": 21},
]
pd.DataFrame(rows)

In [13]:
# 3) From a numpy array
arr = np.random.randint(1, 100, size=(4, 3))
pd.DataFrame(arr, columns=["A", "B", "C"])

In [14]:
### 1.3 Exploring a DataFrame

In [15]:
print(df.shape)      # (rows, columns)
print(df.columns.tolist())
print(df.index.tolist())
print(df.dtypes)

In [16]:
df.head(3)      # first 3 rows

In [17]:
df.tail(2)      # last 2 rows

In [18]:
df.sample(2)    # 2 random rows

In [19]:
df.info()       # column names, non-null counts, dtypes, memory usage

In [20]:
df.describe()   # summary statistics for numeric columns

In [21]:
> 🧠 **Tip:** `.info()` tells you about *structure* (types, missing values). `.describe()` tells you about *distribution* (mean, std, min/max, quartiles). Always run both when you get a new dataset.

In [22]:
# Part 2 — Reading & Writing Data

In real projects, data usually comes from files (CSV, Excel, JSON, SQL) rather than being typed by hand.

In [23]:
# Save our DataFrame to a CSV file, then read it back
df.to_csv("students.csv", index=False)

df_csv = pd.read_csv("students.csv")
df_csv

In [24]:
# JSON
df.to_json("students.json", orient="records")
df_json = pd.read_json("students.json")
df_json

In [25]:
Other common readers/writers you'll use outside this notebook:

```python
pd.read_excel("file.xlsx", sheet_name="Sheet1")
df.to_excel("file.xlsx", index=False)

pd.read_sql("SELECT * FROM table", connection)

pd.read_csv("data.csv", sep=";", encoding="utf-8", na_values=["NA", "?"])
```

In [26]:
# Converting to other Python structures
print(df.to_dict(orient="records")[:2])
print()
print(df.to_numpy()[:2])

In [27]:
# Part 3 — Selection & Indexing

This is the single most-used skill in pandas: picking the rows/columns you need.

In [28]:
### 3.1 Selecting columns

In [29]:
df["name"]              # single column -> Series

In [30]:
df[["name", "marks"]]   # multiple columns -> DataFrame

In [31]:
### 3.2 Selecting rows — `.loc` (label-based) vs `.iloc` (position-based)

In [32]:
df.loc[0]            # row with index label 0

In [33]:
df.loc[0:2]           # rows with labels 0,1,2 (inclusive!)

In [34]:
df.iloc[0:2]          # rows at positions 0,1 (exclusive of 2, like normal Python slicing)

In [35]:
df.loc[0:2, ["name", "city"]]   # rows 0-2, specific columns

In [36]:
df.iloc[1, 0]          # value at row-position 1, column-position 0

In [37]:
### 3.3 Boolean (conditional) indexing

In [38]:
df[df["marks"] > 85]

In [39]:
# Multiple conditions: use & (and), | (or), ~ (not) — and wrap each condition in ( )
df[(df["marks"] > 80) & (df["city"] == "Delhi")]

In [40]:
# .query() -- often more readable for complex conditions
df.query("marks > 80 and city == 'Delhi'")

In [41]:
### 3.4 Adding, modifying, dropping columns

In [42]:
df["grade"] = np.where(df["marks"] >= 90, "A", np.where(df["marks"] >= 80, "B", "C"))
df

In [43]:
df.drop(columns=["grade"])     # returns a NEW dataframe (original untouched unless inplace=True)

In [44]:
### 3.5 Setting a custom index

In [45]:
df_indexed = df.set_index("name")
df_indexed

In [46]:
df_indexed.loc["Priya"]        # now you can select rows by name directly

In [47]:
df_indexed.reset_index()       # undo set_index, back to default 0..n index

In [48]:
# Part 4 — Data Cleaning

Real-world data is messy: missing values, duplicates, wrong types, inconsistent text. Let's build a messy dataset on purpose and clean it.

In [49]:
messy = pd.DataFrame({
    "name": ["Amit", "Priya", "Rahul", "Rahul", "Sneha", None],
    "age": [23, np.nan, 25, 25, 22, 30],
    "city": ["Delhi", "mumbai", "Pune", "Pune", "DELHI", "Chennai"],
    "salary": ["50000", "62,000", "58000", "58000", "N/A", "71000"]
})
messy

In [50]:
### 4.1 Detecting missing values

In [51]:
print(messy.isna().sum())     # count of missing values per column
messy.isna()                  # boolean mask, True = missing

In [52]:
### 4.2 Handling missing values

In [53]:
messy.dropna(subset=["name"])          # drop rows where 'name' is missing

In [54]:
messy_filled = messy.copy()
messy_filled["age"] = messy_filled["age"].fillna(messy_filled["age"].median())
messy_filled

In [55]:
### 4.3 Removing duplicates

In [56]:
print(messy.duplicated().sum())        # how many fully duplicate rows?
messy.drop_duplicates()

In [57]:
### 4.4 Fixing text inconsistencies

In [58]:
messy_clean = messy.copy()
messy_clean["city"] = messy_clean["city"].str.strip().str.title()
messy_clean

In [59]:
### 4.5 Fixing data types

In [60]:
messy_clean["salary"] = (
    messy_clean["salary"]
    .replace("N/A", np.nan)
    .str.replace(",", "", regex=False)
    .astype(float)
)
messy_clean.dtypes

In [61]:
### 4.6 Renaming columns & replacing values

In [62]:
messy_clean = messy_clean.rename(columns={"salary": "salary_inr"})
messy_clean["city"] = messy_clean["city"].replace({"Delhi": "New Delhi"})
messy_clean

In [63]:
> 🧠 **Cleaning checklist for any new dataset:**
> 1. `.info()` → check dtypes & missing counts
> 2. `.duplicated().sum()` → check duplicates
> 3. `.describe()` → check for impossible values (negative age, etc.)
> 4. Standardize text (`.str.strip()`, `.str.lower()`/`.str.title()`)
> 5. Fix dtypes (`astype`) *after* cleaning text representations of numbers

In [64]:
# Part 5 — Data Manipulation

Transforming columns, applying custom functions, sorting, binning, and working with text/dates.

In [65]:
### 5.1 `apply()`, `map()`

In [66]:
df["marks_percent"] = df["marks"].apply(lambda x: f"{x}%")
df

In [67]:
# map() works element-wise on a Series, great for simple lookups
city_region = {"Delhi": "North", "Mumbai": "West", "Pune": "West", "Chennai": "South"}
df["region"] = df["city"].map(city_region)
df

In [68]:
# apply() on a whole row (axis=1) to combine multiple columns
df["summary"] = df.apply(lambda row: f"{row['name']} ({row['age']}) - {row['marks']} marks", axis=1)
df[["name", "summary"]]

In [69]:
### 5.2 Sorting

In [70]:
df.sort_values("marks", ascending=False)

In [71]:
df.sort_values(["city", "marks"], ascending=[True, False])   # sort by multiple columns

In [72]:
### 5.3 Ranking

In [73]:
df["rank"] = df["marks"].rank(ascending=False).astype(int)
df.sort_values("rank")

In [74]:
# Part 6 — GroupBy & Aggregation

`groupby()` follows the **split → apply → combine** pattern: split data into groups, apply a function to each group, combine the results.

In [75]:
sales = pd.DataFrame({
    "region": ["North", "South", "North", "West", "South", "West", "North"],
    "product": ["A", "B", "A", "A", "A", "B", "B"],
    "units": [10, 5, 8, 12, 7, 9, 4],
    "revenue": [1000, 500, 800, 1200, 700, 900, 400]
})
sales

In [76]:
### 6.1 Basic groupby + aggregation

In [77]:
sales.groupby("region")["revenue"].sum()

In [78]:
sales.groupby("region")["revenue"].mean().round(2)

In [79]:
### 6.2 Multiple aggregations with `.agg()`

In [80]:
sales.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_units=("units", "mean"),
    orders=("units", "count")
).reset_index()

In [81]:
### 6.3 Grouping by multiple columns

In [82]:
sales.groupby(["region", "product"])["revenue"].sum().reset_index()

In [83]:
### 6.4 `transform()` — add group stats back onto every row

In [84]:
sales["region_avg_revenue"] = sales.groupby("region")["revenue"].transform("mean")
sales

In [85]:
### 6.5 `filter()` — keep only groups matching a condition

In [86]:
sales.groupby("region").filter(lambda g: g["revenue"].sum() > 1500)

In [87]:
# Part 8 — Reshaping Data: `pivot_table`, `melt`, `crosstab`

Reshaping switches data between **wide** format (one row per entity, many columns) and **long/tidy** format (one row per observation).

In [88]:
### 8.1 `pivot_table` — long → wide, with aggregation

In [89]:
sales.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum", fill_value=0)

In [90]:
### 8.2 `melt` — wide → long

In [91]:
wide = pd.DataFrame({
    "student": ["Amit", "Priya"],
    "math": [80, 90],
    "science": [75, 95]
})
wide.melt(id_vars="student", var_name="subject", value_name="score")